# 🔁 06_orquestrador — Execução contínua do pipeline (polling)
**Squad 2 · Dupla 3**

Notebook "gerente": não processa dados. Em laço, chama os notebooks do pipeline, lê o resumo de cada um, registra cada ciclo em `metadata/`
e encerra sozinho quando o gerador para de enviar arquivos.

| Etapa | Conteúdo |
|---|---|
| 1 | Configuração e parâmetros do laço |
| 2 | Execução de uma etapa do pipeline |
| 3 | Laço de polling |
| 4 | Relatório da execução e conferência final da Bronze |

O pipeline é Bronze → Silver → Gold. A Bronze roda em todo ciclo; Silver e Gold só quando ela traz arquivo novo, no primeiro ciclo ou para refazer um ciclo que teve erro (`SEMPRE_RODAR_TUDO = True` volta a rodar tudo sempre).
Para parar antes do fim: botão **Cancel** (o registro em `metadata/` é salvo a cada ciclo).

**Uma execução por vez:** o laço só começa se conseguir criar a trava `metadata/ecommerce_pedidos/orquestrador_trava.json`. Dois orquestradores ao mesmo tempo leriam os mesmos arquivos novos e gravariam a Bronze em dobro (regra 2). Uma trava sem sinal de vida há mais de 30 min é de uma execução que caiu e é descartada.

## Etapa 1 — Configuração
**Faz:** carrega o `00_setup_config` (para o registro em `metadata/` e a conferência final) e define os parâmetros do laço.
**Valida:** se os parâmetros são coerentes.

In [0]:
VALIDACAO_COMPLETA = False

In [0]:
%run ./00_setup_config

In [0]:
import time
from pyspark.sql import functions as F

PIPELINE = ["./03_ingestao_bronze", "./04_ingestao_silver", "./05_ingestao_gold"]   # ordem de execução
INTERVALO_S = 60                         # espera entre ciclos
MAX_CICLOS_SEM_DADOS = 3                 # encerra após N ciclos seguidos sem arquivo novo (gerador parado)
SEMPRE_RODAR_TUDO = False                # False: Silver e Gold só rodam quando a Bronze traz novidade (ou para refazer um ciclo com erro)
MAX_CICLOS = 180                         # limite de segurança (~3 h com intervalo de 60 s)
TIMEOUT_S = 900                          # tempo máximo de cada notebook chamado
INICIO = datetime.now(timezone.utc)
ANOTACAO_LOG = f"orquestrador_{INICIO:%Y%m%d_%H%M%S}"   # metadata/ecommerce_pedidos/orquestrador_AAAAMMDD_HHMMSS.json

TRAVA = "orquestrador_trava"               # metadata/ecommerce_pedidos/orquestrador_trava.json
TRAVA_EXPIRA_MIN = 30                    # trava sem sinal de vida há mais que isso = execução que caiu


def travar():
    """Cria a trava de forma atômica (falha se já existe). Devolve None se conseguiu, ou o dono da trava ativa."""
    from azure.core.exceptions import ResourceExistsError
    cliente = fs_squad.get_file_client(caminho_metadata(TRAVA))
    dono = {"execucao": ANOTACAO_LOG, "sinal_de_vida_utc": datetime.now(timezone.utc).isoformat()}
    for _ in range(2):
        try:
            cliente.upload_data(json.dumps(dono), overwrite=False)
            return None
        except ResourceExistsError:
            atual = ler_metadata(TRAVA) or {}
            idade = datetime.now(timezone.utc) - datetime.fromisoformat(atual.get("sinal_de_vida_utc", "1970-01-01T00:00:00+00:00"))
            if idade.total_seconds() < TRAVA_EXPIRA_MIN * 60:
                return atual
            cliente.delete_file()                # trava abandonada: descarta e tenta de novo
    return {"execucao": "desconhecida"}


def renovar_trava():
    escrever_metadata(TRAVA, {"execucao": ANOTACAO_LOG, "sinal_de_vida_utc": datetime.now(timezone.utc).isoformat()})


def destravar():
    if (ler_metadata(TRAVA) or {}).get("execucao") == ANOTACAO_LOG:   # só libera a própria trava
        fs_squad.get_file_client(caminho_metadata(TRAVA)).delete_file()


# ------------------------------ validação ------------------------------
checar(PIPELINE and PIPELINE[0].endswith("bronze"), f"Pipeline: {' → '.join(PIPELINE)}", "A primeira etapa do pipeline deve ser a Bronze")
checar(INTERVALO_S > 0 and 0 < MAX_CICLOS_SEM_DADOS <= MAX_CICLOS, "Parâmetros do laço coerentes", "Parâmetros do laço inválidos")
print(f"Intervalo {INTERVALO_S} s · para após {MAX_CICLOS_SEM_DADOS} ciclos sem dados ou {MAX_CICLOS} ciclos · registro: {caminho_metadata(ANOTACAO_LOG)}")

## Etapa 2 — Execução de uma etapa
**Faz:** chama um notebook com `dbutils.notebook.run` (ambiente separado: uma falha não derruba o laço) e devolve o resumo JSON que ele publicou, ou o erro.
Quando o Databricks devolve só "Workload failed", a causa vem da anotação `ultimo_erro` que a validação do notebook gravou ao parar.
**Valida:** a função existe e devolve sempre o mesmo formato.

In [0]:
def causa_do_erro(texto):
    """O erro de dbutils.notebook.run vem embrulhado (WorkflowException); procura a parte que explica a causa real."""
    for marca in ("❌", "AssertionError", "AnalysisException", "Error:", "Exception:"):
        posicao = texto.rfind(marca)
        if posicao >= 0:
            return texto[posicao:posicao + 300].strip()
    return texto[-300:].strip()


def executar(notebook):
    """Roda um notebook do pipeline e devolve {'status', 'duracao_s', 'resumo' ou 'erro'}."""
    inicio = time.time()
    try:
        resumo = json.loads(dbutils.notebook.run(notebook, timeout_seconds=TIMEOUT_S) or "{}")
        return {"status": "OK", "duracao_s": round(time.time() - inicio, 1), "resumo": resumo}
    except Exception as erro:            # o erro fica registrado; o próximo ciclo tenta de novo
        texto, causa = str(erro), causa_do_erro(str(erro))
        # O envelope do Databricks pode vir sem a mensagem do notebook: usa a anotação que o checar() gravou ao parar
        anotado = ler_metadata("ultimo_erro") or {}
        if anotado.get("momento_utc", "") >= datetime.fromtimestamp(inicio, timezone.utc).isoformat():
            causa = anotado["mensagem"]
        return {"status": "ERRO", "duracao_s": round(time.time() - inicio, 1), "erro": causa, "erro_completo": f"{causa}\n{texto[-1500:]}"}


# ------------------------------ validação ------------------------------
checar(callable(executar), "Função de execução pronta", "Função de execução indisponível")

## Etapa 3 — Laço de polling
**Faz:** a cada ciclo, roda a Bronze e, se houver novidade, a Silver e a Gold; imprime uma linha de acompanhamento e salva o histórico em `metadata/`.
Se uma etapa falhar, as seguintes do ciclo não rodam (dependem dela), e o próximo ciclo refaz tudo. Conta os ciclos sem arquivo novo e encerra no limite.
**Valida:** a trava foi obtida (nenhum outro orquestrador rodando); cada ciclo é registrado; erros aparecem na linha do ciclo e no relatório da Etapa 4.

In [0]:
historico, ciclo, sem_dados = [], 0, 0
refazer = True                           # o primeiro ciclo sempre roda tudo, para colocar o pipeline em dia
erros_mostrados = {}                     # notebook -> último erro já impresso por completo
outro = travar()
checar(outro is None, f"Trava obtida: só esta execução processa ({ANOTACAO_LOG})",
       f"Outro orquestrador está rodando ({outro and outro.get('execucao')}, sinal de vida {outro and outro.get('sinal_de_vida_utc')}): "
       "espere ele terminar ou pare o outro antes")
print(f"Início: {INICIO:%Y-%m-%d %H:%M:%S} UTC")
try:
    while ciclo < MAX_CICLOS and sem_dados < MAX_CICLOS_SEM_DADOS:
        ciclo += 1
        renovar_trava()
        registro = {"ciclo": ciclo, "inicio_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")}
        registro[PIPELINE[0]] = executar(PIPELINE[0])          # a Bronze sempre roda: é ela que descobre arquivos novos
        bronze = registro[PIPELINE[0]]
        novos = bronze.get("resumo", {}).get("arquivos_gravados", 0)

        # Silver e Gold só trabalham se há o que processar; sem novidade, o ciclo termina em segundos
        rodar_resto = bronze["status"] == "OK" and (SEMPRE_RODAR_TUDO or novos or refazer)
        for notebook in PIPELINE[1:]:
            if not rodar_resto:
                registro[notebook] = {"status": "PULADO", "duracao_s": 0}
                continue
            registro[notebook] = executar(notebook)
            if registro[notebook]["status"] == "ERRO":
                break                                          # etapas seguintes dependem desta
        refazer = any(registro.get(n, {}).get("status") == "ERRO" for n in PIPELINE)   # erro: o próximo ciclo roda tudo de novo
        sem_dados = 0 if novos else sem_dados + 1
        historico.append(registro)
        escrever_metadata(ANOTACAO_LOG, {"inicio_utc": INICIO.isoformat(), "pipeline": PIPELINE, "ciclos": historico})

        detalhe = (f"Bronze: {novos} arquivos · {bronze['resumo'].get('linhas_gravadas', 0)} linhas · {bronze['resumo'].get('rejeitados', 0)} rejeitados"
                   if bronze["status"] == "OK" else f"❌ Bronze: {bronze['erro'][:120]}")
        silver = registro.get("./04_ingestao_silver")
        if silver and silver["status"] == "PULADO":
            detalhe += " | Silver/Gold: sem novidades"
            silver = None
        if silver:
            detalhe += (f" | Silver: {silver['resumo'].get('aprovadas', 0)} aprovadas · {silver['resumo'].get('quarentena', 0)} quarentena"
                        if silver["status"] == "OK" else f" | ❌ Silver: {silver['erro'][:120]}")
        gold = registro.get("./05_ingestao_gold")
        if gold and gold["status"] != "PULADO":
            detalhe += (f" | Gold: {gold['resumo'].get('alertas_novos', 0)} alertas novos"
                        if gold["status"] == "OK" else f" | ❌ Gold: {gold['erro'][:120]}")
        duracao = round(sum(registro[n]["duracao_s"] for n in PIPELINE if n in registro), 1)
        print(f"[{datetime.now(timezone.utc):%H:%M:%S}] ciclo {ciclo:>3} · {detalhe} · {duracao} s · sem dados: {sem_dados}/{MAX_CICLOS_SEM_DADOS}")
        for notebook in PIPELINE:                              # erro novo: mostra o texto completo uma vez, logo abaixo do ciclo
            resultado = registro.get(notebook, {})
            if resultado.get("status") == "ERRO" and erros_mostrados.get(notebook) != resultado["erro"]:
                erros_mostrados[notebook] = resultado["erro"]
                print(f"    ↳ {notebook}: {resultado['erro_completo']}")
        if sem_dados < MAX_CICLOS_SEM_DADOS:
            time.sleep(INTERVALO_S)
except KeyboardInterrupt:
    print("⏹ Interrompido manualmente (o histórico até aqui está salvo em metadata/)")
finally:
    destravar()                                  # libera a trava mesmo em erro ou cancelamento

motivo_fim = ("sem dados novos" if sem_dados >= MAX_CICLOS_SEM_DADOS else "limite de ciclos" if ciclo >= MAX_CICLOS else "interrupção manual")
print(f"Fim: {ciclo} ciclos · motivo: {motivo_fim}")

## Etapa 4 — Relatório da execução
**Faz:** monta a tabela dos ciclos e confere a Bronze inteira ao final.
**Valida:** nenhum ciclo com erro, todos os arquivos do `raw` na Bronze (ou anotados como rejeitados), nenhum arquivo ingerido mais de uma vez (regra 2). `id_pedido` repetido na Silver é só monitorado (⚠️): a planilha não pede deduplicação por pedido.

In [0]:
etapa_bronze = PIPELINE[0]
def campo(r, etapa, chave):
    """Valor do resumo de uma etapa num ciclo (0 se a etapa não rodou ou falhou)."""
    return r.get(etapa, {}).get("resumo", {}).get(chave, 0)

relatorio = pd.DataFrame([{
    "ciclo": r["ciclo"], "inicio_utc": r["inicio_utc"],
    "status": " / ".join(r[e]["status"] for e in PIPELINE if e in r),
    "bronze_arquivos": campo(r, etapa_bronze, "arquivos_gravados"), "bronze_linhas": campo(r, etapa_bronze, "linhas_gravadas"),
    "bronze_rejeitados": campo(r, etapa_bronze, "rejeitados"),
    "silver_aprovadas": campo(r, "./04_ingestao_silver", "aprovadas"), "silver_quarentena": campo(r, "./04_ingestao_silver", "quarentena"),
    "gold_alertas_novos": campo(r, "./05_ingestao_gold", "alertas_novos"),
    "duracao_s": round(sum(r[e]["duracao_s"] for e in PIPELINE if e in r), 1),
    "erro": " | ".join(r[e].get("erro", "") for e in PIPELINE if e in r and r[e]["status"] == "ERRO")} for r in historico])

bronze = ler_delta(CAMINHO_BRONZE)
por_arquivo = bronze.groupBy("bronze_source_file").agg(F.countDistinct("bronze_ingested_at").alias("ingestoes"))
arquivos_bronze = {r["bronze_source_file"] for r in por_arquivo.select("bronze_source_file").collect()}
faltando = set(listar_arquivos_origem()) - arquivos_bronze - set(ler_metadata("bronze_ignorados") or {})

# ------------------------------ validação ------------------------------
checar(not relatorio["erro"].astype(bool).any(), "Nenhum ciclo com erro", f"{int(relatorio['erro'].astype(bool).sum())} ciclos com erro", parar=False)
checar(not faltando, "Todo arquivo do raw está na Bronze ou anotado como rejeitado",
       f"{len(faltando)} arquivos ainda não ingeridos (chegaram após o último ciclo?)", parar=False)
checar(por_arquivo.filter(F.col("ingestoes") > 1).count() == 0, "Nenhum arquivo ingerido mais de uma vez (regra 2)",
       "Há arquivos ingeridos mais de uma vez")
if tabela_delta_existe(CAMINHO_SILVER):
    checar(ler_delta(CAMINHO_SILVER).groupBy("id_pedido").count().filter("count > 1").count() == 0,
           "Nenhum id_pedido repetido na Silver", "Há id_pedido repetidos na Silver (fora da planilha: só monitorado)", parar=False)

# ------------------------------ resultado ------------------------------
display(relatorio) if len(relatorio) else print("Nenhum ciclo concluído.")   # display() não aceita tabela vazia
print(f"Ciclos: {len(relatorio)} · arquivos na Bronze: {int(relatorio['bronze_arquivos'].sum())} · linhas: {int(relatorio['bronze_linhas'].sum())} · "
      f"Silver: {int(relatorio['silver_aprovadas'].sum())} aprovadas, {int(relatorio['silver_quarentena'].sum())} quarentena · "
      f"Gold: {int(relatorio['gold_alertas_novos'].sum())} alertas · "
      f"Bronze total: {len(arquivos_bronze)} micro-lotes · registro: {CONTAINER_SQUAD}/{caminho_metadata(ANOTACAO_LOG)}")